# Phase 1 Baseline — Run Record


In [1]:
import yaml
from pathlib import Path

config_path = Path("../../config/baseline.yaml")
print(yaml.safe_load(config_path.read_text()))


{'run': {'n_runs': 20, 'run_id_prefix': 'baseline'}, 'llm': {'model': 'openai/gpt-4o-mini', 'base_url': '${OPENAI_BASE_URL}', 'api_key': '${OPENAI_API_KEY}', 'temperature': 0.0, 'max_tokens': 2048, 'request_timeout': 120}, 'mcp': {'workspace_root': 'workspace/'}, 'artifacts': {'output_dir': 'artifacts/'}}


In [4]:
import asyncio
import sys
import nest_asyncio
nest_asyncio.apply()  # Required: Jupyter already has a running event loop

from dotenv import load_dotenv
load_dotenv(Path("../../.env"))

sys.path.insert(0, str(Path("../../src")))
sys.path.insert(0, str(Path("../../")))

from aip_intern.core.config import load_config
from aip_intern.baseline.runner import RunConfig, run

app_cfg = load_config(config_path)
run_cfg = RunConfig(
    run_id_prefix=app_cfg.run.run_id_prefix,
    n_runs=app_cfg.run.n_runs,
    config_path=config_path,
    llm_model=app_cfg.llm.model,
    llm_base_url=app_cfg.llm.base_url,
    llm_api_key=app_cfg.llm.api_key,
    workspace_root=Path("../../workspace/"),
    artifacts_dir=Path("../../artifacts/"),
)

# TODO: execute before final commit — requires OPENAI_BASE_URL and OPENAI_API_KEY to be set
results = asyncio.run(run(run_cfg))
print(f"\n{sum(r.success for r in results)}/{len(results)} runs succeeded")


  Run 1/20... OK
  Run 2/20... OK
  Run 3/20... OK
  Run 4/20... OK
  Run 5/20... OK
  Run 6/20... OK
  Run 7/20... OK
  Run 8/20... OK
  Run 9/20... OK
  Run 10/20... OK
  Run 11/20... OK
  Run 12/20... OK
  Run 13/20... OK
  Run 14/20... OK
  Run 15/20... OK
  Run 16/20... OK
  Run 17/20... OK
  Run 18/20... OK
  Run 19/20... OK
  Run 20/20... OK

20/20 runs succeeded


In [6]:
import pandas as pd

# Show triage.csv from the last successful run
last_success = next((r for r in reversed(results) if r.success), None)
if last_success:
    triage_path = last_success.outputs_path / "triage.csv"
    if triage_path.exists():
        # engine='python' is more forgiving; skip rows the parser can't tokenize
        display(pd.read_csv(triage_path, engine="python", on_bad_lines="skip"))


,id,category,urgency,owner,summary,pii_flagged
0,msg_001,Infrastructure,HIGH,Public Works Dept,Dangerous pothole on Maple Street has caused d...,TRUE
1,msg_003,Noise & Nuisance,MEDIUM,Community Standards,Construction noise at 22 Oak Road is violating...,LOW


In [1]:
from IPython.display import Markdown
if last_success:
    brief_path = last_success.outputs_path / "brief.md"
    if brief_path.exists():
        display(Markdown(brief_path.read_text()))


NameError: name 'last_success' is not defined

In [ ]:
import json
if last_success:
    metrics_path = last_success.outputs_path.parent / "metrics.json"
    if metrics_path.exists():
        print(json.dumps(json.loads(metrics_path.read_text()), indent=2))
